# Notebook 02: Baseline Model Development

## HIV Drug Resistance Prediction with ESM-2

---

**Objective**: Develop baseline model using binary mutation encoding.

**Approach**:
- Binary mutation encoding (1 if mutated from reference, 0 otherwise)
- XGBoost classifier per drug
- 5-fold stratified cross-validation

**Target**: Mean AUC ~0.955 (baseline for comparison)

---

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data_processing import load_fasta, get_drug_list, load_unified_data
from feature_engineering import (
    create_binary_mutation_encoding,
    HIV_PROTEASE_REFERENCE, HIV_RT_REFERENCE
)
from models import train_xgboost, per_drug_training, aggregate_drug_results
from evaluation import compute_auc, stratified_cv, bootstrap_auc
from visualization import plot_roc_curves, plot_drug_comparison

# Random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Imports complete")

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")

## 1. Load Data

In [ ]:
# Load unified data
unified_data = load_unified_data(DATA_DIR)

print("\nLoaded data:")
for dc, data in unified_data.items():
    print(f"  {dc}: {len(data['sequences'])} sequences")

## 2. Create Binary Mutation Encoding

Encode each position as 1 (mutated from reference) or 0 (same as reference).

In [ ]:
# Reference sequences
references = {
    'PI': HIV_PROTEASE_REFERENCE,
    'NRTI': HIV_RT_REFERENCE,
    'NNRTI': HIV_RT_REFERENCE
}

print("Reference sequences:")
for dc, ref in references.items():
    print(f"  {dc}: {len(ref)} aa")

In [ ]:
# Create binary encodings
encodings = {}

for drug_class, data in unified_data.items():
    print(f"\nEncoding {drug_class}...")
    
    sequences = data['sequences']
    reference = references[drug_class]
    
    encoding = create_binary_mutation_encoding(sequences, reference)
    encodings[drug_class] = encoding
    
    # Statistics
    mutation_rate = encoding.mean(axis=0)
    print(f"  Shape: {encoding.shape}")
    print(f"  Mean mutations per sequence: {encoding.sum(axis=1).mean():.1f}")
    print(f"  Most mutated positions: {np.argsort(mutation_rate)[-5:][::-1] + 1}")

## 3. Train Baseline Models (XGBoost)

Train per-drug classifiers using XGBoost with 5-fold cross-validation.

In [ ]:
# Train baseline models for each drug class
baseline_results = {}

for drug_class, data in unified_data.items():
    print(f"\n{'='*60}")
    print(f"TRAINING {drug_class} BASELINE")
    print(f"{'='*60}")
    
    X = encodings[drug_class]
    phenotypes = data['phenotypes']
    drugs = get_drug_list(drug_class)
    
    results = per_drug_training(
        X, phenotypes, drugs,
        model_type='xgboost',
        n_splits=5,
        random_state=RANDOM_SEED
    )
    
    baseline_results[drug_class] = results

## 4. Results Summary

In [ ]:
# Aggregate results
print("\n" + "="*60)
print("BASELINE RESULTS SUMMARY")
print("="*60)

all_aucs = []

for drug_class, results in baseline_results.items():
    print(f"\n{drug_class}:")
    summary = aggregate_drug_results(results)
    print(summary[['drug', 'auc', 'n_samples']].to_string(index=False))
    
    aucs = [r['auc'] for r in results.values()]
    all_aucs.extend(aucs)

print(f"\n{'='*60}")
print(f"Overall Mean AUC: {np.mean(all_aucs):.4f} (+/- {np.std(all_aucs):.4f})")
print(f"Median AUC: {np.median(all_aucs):.4f}")
print(f"Range: [{np.min(all_aucs):.4f}, {np.max(all_aucs):.4f}]")

In [ ]:
# Plot ROC curves for each drug class
for drug_class, results in baseline_results.items():
    fig = plot_roc_curves(
        results,
        title=f"Baseline ROC Curves - {drug_class}",
        save_path=FIGURES_DIR / f'baseline_roc_{drug_class}.png'
    )
    plt.show()

## 5. Save Baseline Results

In [ ]:
import pickle

# Save results for comparison with ESM-2
baseline_path = RESULTS_DIR / 'baseline_results.pkl'

with open(baseline_path, 'wb') as f:
    pickle.dump(baseline_results, f)

print(f"Saved baseline results to: {baseline_path}")

# Save as CSV for easy viewing
all_results = []
for drug_class, results in baseline_results.items():
    for drug, res in results.items():
        all_results.append({
            'drug_class': drug_class,
            'drug': drug,
            'auc': res['auc'],
            'n_samples': res['n_samples'],
            'n_resistant': res['n_resistant'],
            'n_susceptible': res['n_susceptible']
        })

results_df = pd.DataFrame(all_results)
results_df.to_csv(RESULTS_DIR / 'baseline_results.csv', index=False)

print(f"Saved baseline summary to: {RESULTS_DIR / 'baseline_results.csv'}")

## Summary

**Baseline Performance:**
- Expected mean AUC: ~0.955
- Using binary mutation encoding (position-wise)
- XGBoost classifier with 5-fold stratified CV

**Next steps:**
- Notebook 03: Extract ESM-2 embeddings
- Notebook 04: Compare ESM-2 approach to this baseline